# ಪಾಠ 02 - ಮைக್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್‌ವರ್ಕ್ ಅನ್ವೇಷಣೆ

**ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್‌ವರ್ಕ್ (MAF)** ಒಂದು ಏಐ ಏಜೆಂಟ್‌ಗಳನ್ನು ರಚಿಸಲು ಏಕೀಕೃತ ಫ್ರೇಮ್‌ವರ್ಕ್ ಆಗಿದೆ. ಇದು ನಾಲ್ಕು ಮುಖ್ಯ ಕಟ್ಟಡ ಘടಕಗಳೊಂದಿಗೆ ಸ್ವಚ್ಚ, ಸಂಯೋಜನೀಯ ವಾಸ್ತುಶಿಲ್ಪವನ್ನು ಒದಗಿಸುತ್ತದೆ:

- **ಕ್ಲೈಂಟ್** – ಏಐ ಮಾದರಿ ಅಂತಿಮಬಿಂದುವಿಗೆ ಸಂಪರ್ಕಿಸಿ ಸಂವಹನ ನಿರ್ವಹಿಸುತ್ತದೆ
- **ಏಜೆಂಟ್** – ಸೂಚನೆಗಳು ಮತ್ತು ಉಪಕರಣ ವ್ಯಾಖ್ಯಾನಗಳೊಂದಿಗೆ ಕ್ಲೈಂಟ್ ಅನ್ನು ಮಡತ ಮಾಡುತ್ತದೆ
- **ಉಪಕರಣಗಳು** – ಮಾದರಿ ಕರೆಮಾಡಬಹುದಾದ ಕಸ್ಟಮ್ ಕಾರ್ಯಗಳೊಂದಿಗೆ ಏಜೆಂಟ್ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ವಿಸ್ತರಿಸುತ್ತವೆ
- **ಸೆಷನ್** – ಬಹು ತಿರುಗಾಟ ಸಂವಹನಗಳಿಗಾಗಿ ಸಂಭಾಷಣಾ ಇತಿಹಾಸವನ್ನು ಸಂರಕ್ಷಿಸುತ್ತದೆ

ಈ ಪಾಠದಲ್ಲಿ, ನಾವು ಈ ಕಲ್ಪನೆಗಳನ್ನು ಬಳಸಿಕೊಂಡು ಗಮ್ಯಸ್ಥಳ ಲಭ್ಯತೆ ಪರಿಶೀಲಿಸುವ **ಪ್ರಯಾಣ ಬುಕ್ಕಿಂಗ್ ಏಜೆಂಟ್**ನ್ನು ರಚಿಸುವೆವು.


## ಸೆಟ್ಅಪ್


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## ಏಜೆಂಟ್ ಫ್ರೆيم್‌ವರ್ಕ್ ವಾಸ್ತುಶಿಲ್ಪವನ್ನು ಅರ್ಥಮಾಡಿಕೊಳ್ಳುವುದು

ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೆಿಮ್‌ವರ್ಕ್ ಒಂದು ಪದರದ ವಾಸ್ತುಶಿಲ್ಪವನ್ನು ಅನುಸರಿಸುತ್ತದೆ:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **ಗ್ರಾಹಕ** – ಒಂದು `AzureAIProjectAgentProvider` ಒಂದು ಅಜ್ಯೂರ್ ಓಪನ್ಎಐ ನಿಯೋಜನೆಯನ್ನು ಸಂಪರ್ಕಿಸುತ್ತದೆ. ಇದು ಪ್ರಾಮಾಣೀಕರಣ, ವಿನಂತಿ ಸ್ವರೂಪೀಕರಣ ಮತ್ತು ಪ್ರತಿಕ್ರಿಯೆ ವಿಶ್ಲೇಷಣೆಯನ್ನು ಕೈಗೊಳ್ಳುತ್ತದೆ.
2. **ಏಜೆಂಟ್** – ಗ್ರಾಹಕದಿಂದ `provider.create_agent()` ಮೂಲಕ ಸೃಷ್ಟಿಸಲಾಗುತ್ತದೆ, ಏಜೆಂಟ್ ಮಾದರಿ ಪ್ರವೇಶವನ್ನು ಸೂಚನೆಗಳ (ಸಿಸ್ಟಂ ಪ್ರಾಂಪ್ಟ್) ಮತ್ತು ಸಾಧನಗಳೊಂದಿಗೆ ಸಂಯೋಜಿಸುತ್ತದೆ.
3. **ಸಾಧನಗಳು** – `@tool` দিয়ে ಅಲಂಕೃತ Python ಕಾರ್ಯಗಳು, ಏಜೆಂಟ್ ಕ್ರಮಸ್ಪಂದಿಸುವ ಅಥವಾ ಡೇಟಾವನ್ನು ಪಡೆಯಲು ಬಳಸುತ್ತದೆ.
4. **ಸೆಷನ್** – ಒಂದು `AgentSession` ವಸ್ತು ( `agent.create_session()` ಮೂಲಕ ಸೃಷ್ಟಿ) ಸಂಭಾಷಣಾ ಇತಿಹಾಸವನ್ನು ಸಂಗ್ರಹಿಸುತ್ತದೆ, ಬಹು-ಪತಿ ಸಂವಾದವನ್ನು ಸಕ್ರಿಯಗೊಳಿಸುತ್ತದೆ ಅಲ್ಲಿ ಏಜೆಂಟ್ ಹಿಂದಿನ ಸੰਦਰಭವನ್ನು ನೆನಪಿಡುತ್ತಾನೆ.

ಪ್ರತಿ ಪದರನ್ನು ಹಂತದಂತೆ ನಿರ್ಮಿಸುತ್ತೇವೆ.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool ಡೆಕೋರೇಟರ್‌ನ್ನು ಬಳಸಿ ಟೂಲ್ಗಳನ್ನು ಸೇರಿಸುವುದು

ಟೂಲ್ಗಳು ಏಜೆಂಟ್‌ಗಳಿಗೆ ಟೆಕ್ಸ್ಟ್ ಉತ್ಪಾದನೆಗಿಂತಲೂ ಅತಿಯಾಗಿ ಕ್ರಿಯೆಗಳು ಕೈಗೊಳ್ಳಲು ಅವಕಾಶ ನೀಡುತ್ತವೆ. `@tool` ಡೆಕೋರೇಟರ್ ಒಂದು ಸಾಮಾನ್ಯ ಪೈಥಾನ್ ಫಂಕ್ಷನ್ ಅನ್ನು ಏಜೆಂಟ್ ಕರೆ ಮಾಡಬಹುದಾದ ವಸ್ತುವಾಗಿಸುತ್ತದೆ.

ಮುಖ್ಯ ಅಂಶಗಳು:
- ಪ್ರತಿ ಪ್ಯಾರಾಮೀಟರ್ ಅನ್ನು ಮಾದರಿ ಅರ್ಥಮಾಡಿಕೊಳ್ಳುವಂತೆ `Annotated[type, "description"]` ಅನ್ನು ಬಳಸಿ.
- ಡಾಕ್ಸ್ಟ್ರಿಂಗ್ ಮಾದರಿ ಕಾಣುವ ಟೂಲ್ ವಿವರಣೆಯಾಗುತ್ತದೆ.
- `approval_mode="never_require"` ಅಂದರೆ ಬಳಕೆದಾರ ಅನುಮೋದನೆ ಇಲ್ಲದೆ ಟೂಲ್ ಸ್ವಯಂಚಾಲಿತವಾಗಿ ನಡೆಯುತ್ತದೆ.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## ಸಾಧನಗಳೊಂದಿಗೆ ಏಜೆಂಟ್ ಸೃಷ್ಟಿಸುವುದು

ಈಗ ನಾವು ಕ್ಲೈಂಟ್, ಸೂಚನೆಗಳು ಮತ್ತು ಸಾಧನಗಳನ್ನು ಏಜೆಂಟ್ ಆಗಿ ಸಂಯೋಜಿಸುತ್ತೇವೆ. `ಸೂಚನೆಗಳು` ವ್ಯವಸ್ಥೆಯ ಪ್ರಾಂಪ್ಟ್‌ ಆಗಿವೆ — ಅವು ಏಜೆಂಟ್‌ನ ವ್ಯಕ್ತಿತ್ವ ಮತ್ತು ವರ್ತನೆಯನ್ನು ನಿರ್ಧರಿಸುತ್ತವೆ.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## ಸೆಷನ್‌ಗಳೊಂದಿಗೆ ಬಹು-ತಿರುವು ಸಂಭಾಷಣೆಗಳು

ಒಂದು `AgentSession` (`agent.create_session()` ಮೂಲಕ ಸೃಜಿಸಲ್ಪಟ್ಟ) ಸಂಭಾಷಣೆಯಲ್ಲಿನ ಎಲ್ಲಾ ಸಂದಾಯಗಳನ್ನು ಟ್ರಾಕ್ ಮಾಡುತ್ತದೆ. ಒಂದೇ ಸೆಷನ್ ಅನ್ನು ಪ್ರತಿಯೊಂದು `agent.run()` ಕರೆಗೂ ಪಾಸ್ಸ್ ಮಾಡುವ ಮೂಲಕ, ಏಜೆಂಟ್ ಸಂಪೂರ್ಣ ಸಂಭಾಷಣಾ ಇತಿಹಾಸಕ್ಕೆ ಪ್ರವೇಶ ಹೊಂದಿದ್ದು, ಹಳೆಯ ಸಂದಾಯಗಳಿಗೆ ಉಲ್ಲೇಖಿಸಬಹುದು.

ನಾವು `tools=[check_destination_availability]` ಅನ್ನು ಪಾಸ್ಸ್ ಮಾಡುತ್ತೇವೆ, ಆದ್ದರಿಂದ ಏಜೆಂಟ್ ಪ್ರತಿ ತಿರುವಿನ ಅವಧಿಯಲ್ಲಿ ನಮ್ಮ ಲಭ್ಯತೆ ಪರಿಶೋಧಕವನ್ನು ಕರೆಸಬಹುದು.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್‌ವುರ್ಕ್‌ನ ನಾಲ್ಕು ಅಸ್ತಿ ಸ್ತಂಭಗಳನ್ನು ಅನ್ವೇಷಿಸಿದ್ದೀರಿ:

| ಸಂಜ್ಞಾನ | ನೀವು ಏನು ಕಲಿತಿರಿ |
|---------|------------------|
| **ಕ್ಲೈಂಟು** | `AzureAIProjectAgentProvider` ಕ್ರೆಡೆಂಚಿಯಲ್-ಆಧಾರಿತ ಪ್ರಾಮಾಣೀಕರಣದೊಂದಿಗೆ Azure OpenAIಗೆ ಸಂಪರ್ಕಿಸುತ್ತದೆ |
| **ಏಜೆಂಟ್** | `provider.create_agent()` ಸೂಚನೆಗಳು ಮತ್ತು ಹೆಸರು ಜೊತೆಗೆ ಮಾದರಿ ಸಂಪರ್ಕವನ್ನು ಕಟ್ಟು bundles ಮಾಡುತ್ತದೆ |
| **ಉಪಕರಣಗಳು** | `@tool` ಅಲಂಕಾರವು ಏಜೆಂಟ್ ಕರೆಮಾಡಲು ಪೈಥಾನ್ ಕಾರ್ಯಗಳನ್ನು ಬಹಿರಂಗ ಮಾಡುತ್ತದೆ |
| **ಸೆಷನ್** | `agent.create_session()` ಬಹು ಬಾರಿಗೆ ಸಂಭಾಷಣಾ ಇತಿಹಾಸವನ್ನು ಕಾಯ್ದಿರಿಸುತ್ತದೆ |

ಈ ನಿರ್ಮಾಣ ಘಟಕಗಳು ಒಟ್ಟಾಗಿ ಸಂಭಾಷಣೆ ನಡೆಸಬಹುದಾದ, ಬಾಹ್ಯ ಕಾರ್ಯಗಳನ್ನು ಕರೆಮಾಡುವ ಮತ್ತು ಸಂಗತಿಗಳನ್ನು ಉಳಿಸುವ ಏಜೆಂಟ್‌ಗಳನ್ನು ಸೃಷ್ಟಿಸುತ್ತವೆ — ನಂತರದ ಪಾಠಗಳಲ್ಲಿ ಹೆಚ್ಚುವರಿ ಏಜೆಂಟಿಕ್ ಮಾದರಿಗಳಿಗೆ ಆಧಾರವಾಗಿದೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ವ್ಯವಹಾರ ಸ್ವೀಕಾರಾತ್ಮಕ ಪತ್ರ**:  
ಈ ದಾಖಲೆ AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ಶುದ್ಧತೆಯತ್ತ ಪ್ರಯತ್ನಿಸಿದರೂ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ತಪ್ಪುಗಳು ಅಥವಾ ನಿಷ್ಪಕ್ಷಪಾತತೆಗಳಿರಬಹುದು ಎಂದು ದಯವಿಟ್ಟು ಗಮನಿಸಿ. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಾಖಲೆ ಆಧಾರಭೂತ ಮೂಲವಾಗಿರುತ್ತದೆ. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವನಿಂದ ಅನುವಾದ ಮಾಡಿಸುವುದು ಶಿಫಾರಸು ಮಾಡಲ್ಪಡುತ್ತದೆ. ಈ ಅನುವಾದದ ಬಳಕೆಯಿಂದ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಘಟನೆಗಳಿಗೆ ಅಥವಾ ಭ್ರಮನಿಗೆ ನಾವು ಜವಾಬ್ದಾರಿಯಾಗುವುದಿಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
